In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional
from flask import Flask, request, jsonify,send_from_directory
import json
import onnxruntime

In [ ]:
# demo code 
app = Flask(__name__)

# global variables
global train_logs 
softmax = nn.Softmax(dim=0)

@app.route('/')
def base():
    return "Base Test is OK" # may return a html file

In [ ]:
# interface for users to upload dataset
@app.route('/dataset', methods=["POST"])
def getDataSet():
    global inputs
    global labels 
    request_content = request.get_json()    
    #print(type(request_content))
    
    if(isinstance(request_content, str)):
        request_content = json.loads(request_content)
        #print(type(request_content))
    
    inputs = request_content["inputs"]
    labels = request_content["labels"]
    
    inputs = torch.FloatTensor(inputs)
    labels = torch.FloatTensor(labels)    
    
    response = app.response_class(
        response=json.dumps(request_content),
        status=200,
        mimetype='application/json'
    )
        
    return response
    
# if dataset is meta format, like image, json, yaml, ...
#   students may implement the approach to save the file or store data into an NoSQL database

In [ ]:
# Make device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"# Enable CUDA if possible
print(f"device is {device}")

class MyModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(MyModel, self).__init__()
        hidden_dim = 10
        self.linear_1 = nn.Linear(2, hidden_dim)
        self.linear_2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, inputs):
      outputs = self.linear_1(inputs).requires_grad_()      
      outputs = self.linear_2(outputs).requires_grad_()
      return outputs
        
# 4. Create an instance of the model and send it to target device
model = MyModel(2,3).to(device)

# Create a loss function
criterion = nn.CrossEntropyLoss()  

# Create an optimizer
learning_rate = 0.1
optimizer = torch.optim.SGD(params=model.parameters(),lr=learning_rate)

print(model.parameters)# Type of parameter object

In [ ]:
def encodeLabel(onehats,dim):
    labelcoding = []
    for code in onehats:
        value = 0
        for idx in range(dim):
            if(code[idx]==1):
                value = idx
        labelcoding.append(value)
    labelcoding = torch.FloatTensor(labelcoding) 
    return labelcoding  

In [ ]:
@app.route("/train", methods=['GET', 'POST'])
def trainModel():
    global train_logs
    global inputs
    global labels
    
    train_logs = []    
    num_epochs = 1000 # Train for longer\
    softmax = nn.Softmax(dim=0)
    
    # Put data to target device
    labels=encodeLabel(labels,3)
    inputs, labels = inputs.to(device), labels.to(device)

    for epoch in range(num_epochs):
        predicts = model(inputs)
        loss = criterion(predicts,labels.long() )  # https://www.cnblogs.com/blogwangwang/p/12018897.html
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Print out what's happening every 10 epochs
        if epoch % 100 == 0:
            train_logs.append(f"Epoch: {epoch} | Loss: {loss:.5f}")
    return "OK"

In [ ]:
@app.route("/trainedModel", methods=['GET'])
def getTrainedModel():
    x = [1,2,3]
    x = torch.FloatTensor(x).to(device)
    model.eval()
    torch.onnx.export(model,x,"trainedModel.onnx")
    
    try:
        return send_from_directory('' , 'trainedModel.onnx', as_attachment=True)
    except FileNotFoundError:
        abort(404)

In [ ]:
@app.route("/log", methods=['GET'])
def getTrainingLog():    
    response = app.response_class(
        response=json.dumps(train_logs),
        status=200,
        mimetype='application/json'
    )
    return response

In [ ]:
@app.route("/inference", methods=['POST'])
def inference():
    request_content = request.get_json() 
    if(isinstance(request_content, str)):
        request_content = json.loads(request_content)
    input_data = request_content["input"]
    input_data = torch.FloatTensor(input_data)
    input_data = input_data.to(device)
    result = softmax(model(input_data)).tolist()
    
    response = app.response_class(
        response=json.dumps(result),
        status=200,
        mimetype='application/json'
    ) 
    return response

In [ ]:
# Run up your Web API Server
if __name__ == '__main__':
    app.run(host='0.0.0.0',port=5000)